# 5.1 · Pizarra: diseñar y acotar

**Objetivo:** el primer paso de un agente no es el código, sino **decidir qué no hace**.

Este vídeo no tiene código: este notebook recoge lo que queda en la pizarra, para tenerlo a mano en los tres siguientes.

## El problema

Cada mañana alguien lee la bandeja de entrada de soporte (`datos/bandeja/`) y los clasifica. Vamos a construir un agente que le ayude: para cada correo, identifica al cliente, comprueba si ya existe una incidencia, clasifica la prioridad, busca el artículo de la base de conocimiento relevante y **propone** una respuesta. **No envía nada**: una persona revisa.

## Alcance

| Hace | NO hace |
|---|---|
| Lee correos de la bandeja | Enviar respuestas al cliente |
| Busca incidencias existentes | Cerrar incidencias |
| Crea incidencias nuevas | Modificar datos de clientes |
| Clasifica la prioridad | Escalar a personas |
| Propone respuesta citando la KB | Inventar soluciones que no estén en la KB |

Cada fila de "NO hace" es una herramienta que **no** se le da o un permiso que se le **niega**. El alcance se impone con permisos, no solo con el prompt.

## Diseño

```
                      ┌────────────────────────────┐
 "Tría 003.txt" ────► │      agente de triaje      │ ────► JSON por correo
                      └──────────────┬─────────────┘
           ┌──────────────┬──────────┼───────────────┬──────────────┐
           ▼              ▼          ▼               ▼              ▼
      leer_correo  buscar_incidencias  crear_incidencia  buscar_kb   subagente
                                                                  "clasificador"
      ─────────────── hook de auditoría (todas las llamadas) ───────────────
```

Salida estructurada, una por correo:

```json
{
  "correo": "003_urgente.txt",
  "es_incidencia": true,
  "cliente": "Hooli",
  "incidencia_id": "INC-016",
  "accion": "creada | existente | ninguna",
  "prioridad": "alta",
  "articulo_kb": "reset-password.md",
  "respuesta_propuesta": "..."
}
```

## Criterios de éxito

Tres casos de prueba concretos. Los reutilizaremos como evals en el bloque 6.

1. **Correo duplicado** (`002_duplicado.txt`) → `accion: "existente"` y **no** se crea incidencia.
2. **Correo urgente de acceso** (`003_urgente.txt`) → `prioridad: "alta"`.
3. **Felicitación** (`005_felicitacion.txt`) → `es_incidencia: false` y no se llama a `crear_incidencia`.

Con esto en la pizarra, el código es casi un trámite.

## La bandeja

Para echar un vistazo a los correos:

In [ ]:
%load_ext autoreload
%autoreload 2

from dotenv import load_dotenv
load_dotenv()

import sys; sys.path.append("../..")   # para importar desde la raíz del taller
from pathlib import Path
from config import MODEL

import herramientas
herramientas.listar_bandeja()

In [ ]:
print(herramientas.leer_correo("002_duplicado.txt"))